# 🧠 Classificação de Tumor Cerebral com CNN
**Dataset:** Brain Tumor Classification (MRI) — Kaggle (Sartaj Bhuvaji)  
**Framework:** PyTorch  
**Abordagem:** CNN do zero  

**Classes:**
- `glioma_tumor`
- `meningioma_tumor`
- `pituitary_tumor`
- `no_tumor`

## 1. Instalação e Configuração

In [ ]:
# Instalar kaggle para baixar o dataset
!pip install kaggle -q

# Verificar GPU disponível
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'GPU disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Download do Dataset via Kaggle API

> **Como configurar a API do Kaggle:**
> 1. Acesse [kaggle.com](https://www.kaggle.com) → Account → API → **Create New Token**
> 2. Faça o upload do arquivo `kaggle.json` na célula abaixo

In [ ]:
from google.colab import files

# Faça upload do kaggle.json aqui
uploaded = files.upload()

In [ ]:
import os

# Configurar credenciais do Kaggle
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# Baixar dataset
!kaggle datasets download -d sartajbhuvaji/brain-tumor-classification-mri --unzip -p /content/brain_tumor

print('\nEstrutura do dataset:')
!find /content/brain_tumor -type d | head -20

## 3. Imports

In [ ]:
import os
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
from torchvision import datasets, transforms

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    accuracy_score,
    f1_score
)
from sklearn.preprocessing import label_binarize

# Reprodutibilidade
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {DEVICE}')

## 4. Configuração dos Caminhos e Hiperparâmetros

In [ ]:
# ─── Caminhos ────────────────────────────────────────────────────────────────
# Ajuste os caminhos conforme a estrutura do seu dataset após o download
TRAIN_DIR = '/content/brain_tumor/Training'
TEST_DIR  = '/content/brain_tumor/Testing'

# ─── Hiperparâmetros ──────────────────────────────────────────────────────────
IMG_SIZE    = 128       # Redimensionar imagens para 128x128
BATCH_SIZE  = 32
NUM_EPOCHS  = 30
LR          = 1e-3      # Learning rate inicial
NUM_CLASSES = 4

CLASS_NAMES = ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']

## 5. Pré-processamento e Data Augmentation

In [ ]:
# Transformações para o conjunto de TREINO (com augmentation)
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),  # MRI é cinza, mas CNN espera 3 canais
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Transformações para TESTE (sem augmentation)
test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Carregar datasets
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
test_dataset  = datasets.ImageFolder(root=TEST_DIR,  transform=test_transforms)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Amostras de treino : {len(train_dataset)}')
print(f'Amostras de teste  : {len(test_dataset)}')
print(f'Classes            : {train_dataset.classes}')

## 6. Análise Exploratória do Dataset

In [ ]:
# Distribuição das classes
train_counts = Counter(train_dataset.targets)
test_counts  = Counter(test_dataset.targets)
classes      = train_dataset.classes

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

for ax, counts, title in zip(axes,
                              [train_counts, test_counts],
                              ['Treino', 'Teste']):
    bars = ax.bar([classes[i] for i in sorted(counts)],
                  [counts[i] for i in sorted(counts)],
                  color=colors)
    ax.set_title(f'Distribuição — {title}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Quantidade de imagens')
    ax.set_xticklabels([classes[i] for i in sorted(counts)], rotation=20, ha='right')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(int(bar.get_height())), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('distribuicao_classes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualizar amostras do dataset
def imshow(img, ax, title=''):
    # Desfazer normalização
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img  = img * std + mean
    img  = img.clamp(0, 1)
    ax.imshow(img.permute(1, 2, 0).numpy(), cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.suptitle('Amostras do Dataset — MRI', fontsize=15, fontweight='bold', y=1.01)

# Pegar uma imagem de cada classe para cada linha
class_indices = {cls: [] for cls in range(NUM_CLASSES)}
for idx, (_, label) in enumerate(train_dataset.samples):
    if len(class_indices[label]) < 4:
        class_indices[label].append(idx)

for row, cls in enumerate(range(NUM_CLASSES)):
    for col, sample_idx in enumerate(class_indices[cls]):
        img, _ = train_dataset[sample_idx]
        imshow(img, axes[row][col],
               title=classes[cls] if col == 0 else '')

plt.tight_layout()
plt.savefig('amostras_dataset.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Arquitetura da CNN

In [ ]:
class BrainTumorCNN(nn.Module):
    """
    CNN para classificação de tumores cerebrais.

    Arquitetura:
        Bloco 1: Conv(3→32)  → BN → ReLU → Conv(32→32)  → BN → ReLU → MaxPool → Dropout
        Bloco 2: Conv(32→64) → BN → ReLU → Conv(64→64)  → BN → ReLU → MaxPool → Dropout
        Bloco 3: Conv(64→128)→ BN → ReLU → Conv(128→128)→ BN → ReLU → MaxPool → Dropout
        Classificador: FC(2048→512) → ReLU → Dropout → FC(512→4)
    """

    def __init__(self, num_classes=4):
        super(BrainTumorCNN, self).__init__()

        # ── Bloco Convolucional 1 ──────────────────────────────────────────────
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),   # 128 → 64
            nn.Dropout2d(p=0.25)
        )

        # ── Bloco Convolucional 2 ──────────────────────────────────────────────
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),   # 64 → 32
            nn.Dropout2d(p=0.25)
        )

        # ── Bloco Convolucional 3 ──────────────────────────────────────────────
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),   # 32 → 16
            nn.Dropout2d(p=0.25)
        )

        # ── Classificador Fully Connected ──────────────────────────────────────
        # Tamanho após 3 MaxPool: 128 // (2^3) = 16 → 128 * 16 * 16 = 32768
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(512, num_classes)
        )

        # Inicialização dos pesos
        self._initialize_weights()

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)


# Instanciar o modelo
model = BrainTumorCNN(num_classes=NUM_CLASSES).to(DEVICE)

# Sumário do modelo
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTotal de parâmetros     : {total_params:,}')
print(f'Parâmetros treináveis   : {trainable_params:,}')

## 8. Função de Perda, Otimizador e Scheduler

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)

# ReduceLROnPlateau: reduz LR quando val_loss para de melhorar
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=4, verbose=True
)

print('Critério  :', criterion)
print('Otimizador:', optimizer)
print('Scheduler :', scheduler)

## 9. Funções de Treino e Avaliação

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted  = outputs.max(1)
        correct       += predicted.eq(labels).sum().item()
        total         += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted  = outputs.max(1)
            correct       += predicted.eq(labels).sum().item()
            total         += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    return epoch_loss, epoch_acc


print('Funções de treino e avaliação definidas.')

## 10. Treinamento

In [ ]:
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss':   [], 'val_acc':   []
}

best_val_loss   = float('inf')
best_model_wts  = copy.deepcopy(model.state_dict())
patience_counter = 0
EARLY_STOP_PATIENCE = 8

print(f'{'Época':>6} | {'Treino Loss':>11} | {'Treino Acc':>10} | {'Val Loss':>9} | {'Val Acc':>8} | {'LR':>8} | Tempo')
print('-' * 75)

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss,   val_acc   = evaluate(model, test_loader, criterion, DEVICE)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    elapsed    = time.time() - start

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # Salvar melhor modelo
    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), 'best_model.pth')
        patience_counter = 0
        flag = '✓'
    else:
        patience_counter += 1
        flag = ''

    print(f'{epoch:>6} | {train_loss:>11.4f} | {train_acc:>10.4f} | {val_loss:>9.4f} | {val_acc:>8.4f} | {current_lr:>8.2e} | {elapsed:.1f}s {flag}')

    # Early stopping
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f'\nEarly stopping ativado na época {epoch}.')
        break

# Carregar o melhor modelo
model.load_state_dict(best_model_wts)
print(f'\nTreinamento concluído. Melhor val_loss: {best_val_loss:.4f}')

## 11. Gráficos de Treinamento

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Histórico de Treinamento', fontsize=14, fontweight='bold')

# Loss
axes[0].plot(epochs_ran, history['train_loss'], 'b-o', markersize=4, label='Treino')
axes[0].plot(epochs_ran, history['val_loss'],   'r-o', markersize=4, label='Validação')
axes[0].set_title('Loss por Época')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs_ran, [a*100 for a in history['train_acc']], 'b-o', markersize=4, label='Treino')
axes[1].plot(epochs_ran, [a*100 for a in history['val_acc']],   'r-o', markersize=4, label='Validação')
axes[1].set_title('Acurácia por Época (%)')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Acurácia (%)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('historico_treinamento.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Avaliação no Conjunto de Teste

In [ ]:
def get_predictions(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs   = F.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


y_true, y_pred, y_probs = get_predictions(model, test_loader, DEVICE)

# Métricas gerais
acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred, average='weighted')

print('=' * 50)
print(f'  Acurácia (teste)       : {acc:.4f} ({acc*100:.2f}%)')
print(f'  F1-Score (weighted)    : {f1:.4f}')
print('=' * 50)
print()
print('Relatório de Classificação:')
print(classification_report(y_true, y_pred, target_names=classes))

## 13. Matriz de Confusão

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Matriz de Confusão', fontsize=14, fontweight='bold')

# Contagens absolutas
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes,
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Valores Absolutos')
axes[0].set_ylabel('Classe Real')
axes[0].set_xlabel('Classe Predita')
axes[0].tick_params(axis='x', rotation=25)
axes[0].tick_params(axis='y', rotation=0)

# Normalizada (%)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens',
            xticklabels=classes, yticklabels=classes,
            ax=axes[1], linewidths=0.5)
axes[1].set_title('Normalizada por Classe Real')
axes[1].set_ylabel('Classe Real')
axes[1].set_xlabel('Classe Predita')
axes[1].tick_params(axis='x', rotation=25)
axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('matriz_confusao.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Curva ROC — One vs Rest

In [ ]:
# Binarizar labels para ROC
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

colors_roc = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
fig, ax = plt.subplots(figsize=(9, 7))

auc_scores = []
for i, (cls, color) in enumerate(zip(classes, colors_roc)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
    auc_val      = roc_auc_score(y_true_bin[:, i], y_probs[:, i])
    auc_scores.append(auc_val)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{cls} (AUC = {auc_val:.3f})')

ax.plot([0,1], [0,1], 'k--', lw=1, label='Aleatório')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('Taxa de Falsos Positivos (FPR)', fontsize=12)
ax.set_ylabel('Taxa de Verdadeiros Positivos (TPR)', fontsize=12)
ax.set_title(f'Curva ROC — One vs Rest\nAUC Médio: {np.mean(auc_scores):.3f}', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('curva_roc.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'AUC por classe:')
for cls, auc in zip(classes, auc_scores):
    print(f'  {cls:<22}: {auc:.4f}')
print(f'  {"Média":<22}: {np.mean(auc_scores):.4f}')

## 15. Predições Visuais — Exemplos do Teste

In [ ]:
def show_predictions(model, dataset, device, n=16):
    model.eval()
    indices = np.random.choice(len(dataset), n, replace=False)

    fig, axes = plt.subplots(4, 4, figsize=(14, 14))
    fig.suptitle('Predições no Conjunto de Teste', fontsize=14, fontweight='bold')

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

    for ax, idx in zip(axes.flatten(), indices):
        img, true_label = dataset[idx]
        input_tensor    = img.unsqueeze(0).to(device)

        with torch.no_grad():
            output    = model(input_tensor)
            prob      = F.softmax(output, dim=1)
            pred      = output.argmax(1).item()
            conf      = prob[0, pred].item()

        # Desfazer normalização para exibir
        display_img = (img * std + mean).clamp(0, 1)
        ax.imshow(display_img.permute(1, 2, 0).numpy(), cmap='gray')

        color = 'green' if pred == true_label else 'red'
        ax.set_title(
            f'Real: {classes[true_label]}\nPred: {classes[pred]} ({conf*100:.1f}%)',
            fontsize=7.5, color=color
        )
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('predicoes_exemplo.png', dpi=150, bbox_inches='tight')
    plt.show()

show_predictions(model, test_dataset, DEVICE)

## 16. Resumo Final dos Resultados

In [ ]:
from sklearn.metrics import precision_score, recall_score

precision = precision_score(y_true, y_pred, average='weighted')
recall    = recall_score(y_true, y_pred, average='weighted')
auc_mean  = np.mean(auc_scores)

print('=' * 55)
print('              RESUMO DOS RESULTADOS')
print('=' * 55)
print(f'  Acurácia              : {acc*100:.2f}%')
print(f'  Precisão (weighted)   : {precision:.4f}')
print(f'  Recall (weighted)     : {recall:.4f}')
print(f'  F1-Score (weighted)   : {f1:.4f}')
print(f'  AUC-ROC (média)       : {auc_mean:.4f}')
print('=' * 55)
print(f'  Melhor val_loss       : {best_val_loss:.4f}')
print(f'  Épocas treinadas      : {len(history["train_loss"])}')
print(f'  Parâmetros do modelo  : {trainable_params:,}')
print('=' * 55)

## 17. Salvar Artefatos

In [ ]:
from google.colab import files

# O melhor modelo já foi salvo como 'best_model.pth' durante o treino
# Baixar todos os artefatos gerados
artifacts = [
    'best_model.pth',
    'distribuicao_classes.png',
    'amostras_dataset.png',
    'historico_treinamento.png',
    'matriz_confusao.png',
    'curva_roc.png',
    'predicoes_exemplo.png'
]

for f in artifacts:
    if os.path.exists(f):
        files.download(f)
        print(f'Baixando: {f}')
    else:
        print(f'Arquivo não encontrado: {f}')